# Phase 3: Controlled Architecture Ablation on Kaggle (NVIDIA CUDA GPU)

**Project:** High-Energy Physics (HEP) Anomaly Detection using Graph Neural Networks on the LHC Olympics 2020 (LHCO2020) Dataset.  
**Scientific Question:** Isolate the individual contributions of:
1. **Graph representation** (Relational $\Delta R < 0.4$ topology vs flat leading-particle MLP)
2. **Physics-informed MET objective** (Conservation constraint head with $\lambda_{\text{MET}} = 0.1$)
3. **Adversarial mass decorrelation** (Gradient Reversal Layer regressing invariant mass with $\lambda_{\text{adv}} = 0.05$)

---

### Models in this Controlled Ablation Campaign
| Model | Family | Architecture | Loss Formulation |
| :--- | :--- | :--- | :--- |
| `baseline_mlp` | `architecture` | `BaselineMLP` (64 &rarr; 32 &rarr; 8 &rarr; 32 &rarr; 64) | $\mathcal{L} = \mathcal{L}_{\text{recon}}$ |
| `gat_standalone` | `architecture` | `NodeGNN` (4-head GAT &rarr; 1-head GAT &rarr; Recon) | $\mathcal{L} = \mathcal{L}_{\text{recon}}$ |
| `gat_met_production` | `architecture` | `NodeGNN` + Global MET head | $\mathcal{L} = \mathcal{L}_{\text{recon}} + 0.1 \times \mathcal{L}_{\text{MET}}$ |
| `gat_met_decorrelation` | `architecture` | `AdversarialNodeGNN` + MET + GRL mass head | $\mathcal{L} = \mathcal{L}_{\text{recon}} + 0.1 \times \mathcal{L}_{\text{MET}} + 0.05 \times \mathcal{L}_{\text{adv}}$ |

---
> **CRITICAL KAGGLE SETTINGS CHECKLIST (Right Panel):**
> 1. **Settings &rarr; Accelerator:** Select `GPU T4 x2` or `GPU P100` (16 GB VRAM).
> 2. **Settings &rarr; Internet:** Must be turned **`Internet On`** (requires phone-verified Kaggle account).
> 3. **Working Directory:** `/kaggle/working` is the writable directory where files and results are saved.


## 1. Clone / Pull Repository & Set Working Directory

In [ ]:
import os

# Ensure clean working directory and latest code from GitHub
REPO_DIR = "/kaggle/working/anomaly-detector-gnn"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/KamitKoul/anomaly-detector-gnn.git {REPO_DIR}

%cd {REPO_DIR}
!git fetch origin main
!git reset --hard origin/main
!git log -n 3 --oneline


## 2. Dataset Setup (Kaggle Dataset Auto-Discovery or Zenodo Direct Download)

The LHCO datasets (~2.5 GB background, ~2.6 GB signal) can either be:
- **Auto-discovered** from an attached Kaggle dataset (`/kaggle/input/...`)
- **Downloaded directly from Zenodo** via high-speed datacenter links in 1-2 minutes.

In [ ]:
import os
import glob

!mkdir -p data

bg_target = "data/events_LHCO2020_backgroundMC_Pythia.h5"
sig_target = "data/W_prime_signal.h5"

def is_valid_dataset(path):
    """Checks if path exists and has valid non-empty HDF5 size (> 100 MB)."""
    return os.path.exists(path) and os.path.getsize(path) > 100_000_000

# 1. Search /kaggle/input in case dataset is attached as Kaggle input
input_dir = "/kaggle/input"
if os.path.exists(input_dir):
    if not is_valid_dataset(bg_target):
        bg_candidates = glob.glob(os.path.join(input_dir, "**/*background*Pythia*.h5"), recursive=True) + \
                        glob.glob(os.path.join(input_dir, "**/*background*.h5"), recursive=True)
        if bg_candidates and is_valid_dataset(bg_candidates[0]):
            print(f"[INFO] Discovered Background dataset in Kaggle input: {bg_candidates[0]}")
            !ln -sf "{bg_candidates[0]}" {bg_target}
            
    if not is_valid_dataset(sig_target):
        sig_candidates = glob.glob(os.path.join(input_dir, "**/*W_prime*.h5"), recursive=True) + \
                         glob.glob(os.path.join(input_dir, "**/*anomalydetection*.h5"), recursive=True)
        if sig_candidates and is_valid_dataset(sig_candidates[0]):
            print(f"[INFO] Discovered Signal dataset in Kaggle input: {sig_candidates[0]}")
            !ln -sf "{sig_candidates[0]}" {sig_target}

# 2. If not found in Kaggle input, download directly from Zenodo
if not is_valid_dataset(bg_target):
    print("[ACTION] Downloading Background MC from Zenodo (record 4536624, ~2.5 GB)...")
    !wget -q --show-progress -c -O {bg_target} "https://zenodo.org/api/records/4536624/files/events_LHCO2020_backgroundMC_Pythia.h5/content"

if not is_valid_dataset(sig_target):
    print("[ACTION] Downloading Signal Dataset from Zenodo (record 4536377, ~2.6 GB)...")
    !wget -q --show-progress -c -O {sig_target} "https://zenodo.org/api/records/4536377/files/events_anomalydetection.h5/content"

print("\n[SUCCESS] Datasets are verified and ready:")
for name, p in [("Background", bg_target), ("Signal", sig_target)]:
    if is_valid_dataset(p):
        print(f"  [VERIFIED] {name}: {p} ({os.path.getsize(p)/(1024**3):.2f} GB)")
    else:
        print(f"  [ERROR] {name}: {p} is missing or incomplete!")


## 3. Automated Environment Verification & Diagnostics

Executes `scripts/setup_kaggle.sh` to install dependencies, verify CUDA GPU acceleration, and run config dry-runs.

In [ ]:
!bash scripts/setup_kaggle.sh


In [ ]:
import os
import sys

# Ensure repository root is on PYTHONPATH
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("[DIAGNOSTIC] Running test suite...")
!python3 -m unittest discover -s tests -p "test_*.py"


## 4. Run Isolated GAT CUDA Smoke Test

Validates forward pass, loss computation, backward pass, and CUDA memory allocation on a mini-batch of 20 events for 1 epoch.

In [ ]:
!python src/train_experiment.py \
    --config configs/architecture/gat_met_production.json \
    --limit 20 \
    --epochs 1 \
    --smoke-test \
    --output-dir experiments/smoke_test/kaggle_gat_smoke

!rm -rf experiments/smoke_test/kaggle_gat_smoke


## 5. Phase 3: Train Controlled Architecture Ablation Variants

Scientific Conditions (Preserved strictly across all 4 models):
- Seed: `42`
- Training Population: `5,000` background events
- Graph Neighborhood: $\Delta R < 0.4$
- Log-compression: $p_T \to \ln(1 + p_T)$
- Optimizer: Adam (lr = `0.001`, batch size = `32`)


### 5.1 Experiment 1: Baseline MLP Autoencoder
- Model: `BaselineMLP` (flat 64D event vector: 32 leading particles $\times$ 2 features $[\ln(1+p_T), \Delta R]$)
- Loss: $\mathcal{L} = \mathcal{L}_{\text{recon}}$
- Epochs: 20

In [ ]:
import os
if os.path.exists("experiments/architecture/baseline_mlp/checkpoint.pt"):
    print("[INFO] Baseline MLP checkpoint already exists. Skipping training.")
else:
    !python src/train_experiment.py --config configs/architecture/baseline_mlp.json


### 5.2 Experiment 2: Standalone GAT Autoencoder
- Model: `NodeGNN` (conv1 4-head GAT &rarr; conv2 1-head GAT &rarr; recon head)
- Loss: $\mathcal{L} = \mathcal{L}_{\text{recon}}$ (no MET prediction head)
- Epochs: 20

In [ ]:
import os
if os.path.exists("experiments/architecture/gat_standalone/checkpoint.pt"):
    print("[INFO] Standalone GAT checkpoint already exists. Skipping training.")
else:
    !python src/train_experiment.py --config configs/architecture/gat_standalone.json


### 5.3 Experiment 3: Production GAT + MET Objective (Reference Baseline)
- Model: `NodeGNN` (Multi-Task GAT autoencoder + global MET regression head)
- Loss: $\mathcal{L} = \mathcal{L}_{\text{recon}} + 0.1 \times \mathcal{L}_{\text{MET}}$
- Epochs: 20

In [ ]:
import os
if os.path.exists("experiments/architecture/gat_met_production/checkpoint.pt"):
    print("[INFO] Production GAT checkpoint already exists. Skipping training.")
else:
    !python src/train_experiment.py --config configs/architecture/gat_met_production.json


### 5.4 Experiment 4: GAT + MET + Adversarial Mass Decorrelation
- Model: `AdversarialNodeGNN` (Multi-Task GAT + MET + GRL mass adversary)
- Loss: $\mathcal{L} = \mathcal{L}_{\text{recon}} + 0.1 \times \mathcal{L}_{\text{MET}} + 0.05 \times \mathcal{L}_{\text{adv}}$
- Epochs: 15

In [ ]:
import os
if os.path.exists("experiments/architecture/gat_met_decorrelation/checkpoint.pt"):
    print("[INFO] Adversarial GAT checkpoint already exists. Skipping training.")
else:
    !python src/train_experiment.py --config configs/architecture/gat_met_decorrelation.json


## 6. Single-Experiment Systematic Evaluation

Evaluates all 4 models on background and genuine $W'$ signal events (`truth == 1`), computing:
- ROC AUC
- Background Rejection ($1 / \text{FPR}$) at 30% and 50% signal efficiency
- Statistical Separation Ratio ($\mu_{\text{sig}} / \mu_{\text{bg}}$)
- Separation Significance ($\sigma$)
- Generates `metrics.json` and `predictions.npz` in each experiment directory.

In [ ]:
import os

# Set EVAL_ALL = True to evaluate on 100% of ALL events in the dataset (~1M events!)
# Set EVAL_ALL = False to evaluate on 20,000 events for quick fast verification
EVAL_ALL = True
FORCE_EVAL = False  # Set to True to re-evaluate even if metrics.json exists

eval_flag = "--all" if EVAL_ALL else "--eval-limit 20000"

experiments = [
    "experiments/architecture/baseline_mlp",
    "experiments/architecture/gat_standalone",
    "experiments/architecture/gat_met_production",
    "experiments/architecture/gat_met_decorrelation",
]

print("[INFO] Initiating systematic evaluation across all ablation models...")
for exp in experiments:
    metrics_file = os.path.join(exp, "metrics.json")
    if os.path.exists(metrics_file) and not FORCE_EVAL:
        print(f"[SKIP] {exp} already evaluated (found {metrics_file}).")
        continue
    print(f"\n---> Evaluating {exp}...")
    !python evaluation/evaluate_single.py --experiment "{exp}" {eval_flag}


## 7. Comparative Analysis & Summary Aggregation

Aggregates the evaluation metrics from all 4 models and generates a comparison table in `comparisons/architecture/summary.md`.

In [ ]:
!python evaluation/compare_experiments.py --family architecture

print("\n--- Comparative Results Summary ---")
!cat comparisons/architecture/summary.md


## 7.1 Standalone Background Mass-Sculpting & 1/JSD Evaluation

Evaluates whether anomaly score threshold cuts (top 10%, 5%, 1%) sculpt the background invariant mass spectrum.
Computes both **Raw Dijet Mass** and **Groomed ($m_{\text{RSD}}$) Mass** smoothness ($1 / \text{JSD}$).

In [ ]:
!python evaluation/evaluate_mass_sculpting.py --family architecture --limit 2000


## 8. Package Results for Direct Download

Creates compressed zip and tar.gz archives of all experiment checkpoints, training logs, metrics, and comparison tables.
You can download them directly from the Kaggle Notebook output panel in the right sidebar!

In [ ]:
import os

OUTPUT_ARCHIVE = "/kaggle/working/phase3_results.tar.gz"
OUTPUT_ZIP = "/kaggle/working/phase3_results.zip"

print("[INFO] Packaging experiment checkpoints, metrics, and comparisons...")
!tar -czf "{OUTPUT_ARCHIVE}" experiments/ comparisons/
!zip -q -r "{OUTPUT_ZIP}" experiments/ comparisons/

print(f"[SUCCESS] Archives created successfully:")
!ls -lh /kaggle/working/phase3_results.*

print("\n[DOWNLOAD INSTRUCTIONS]:")
print("1. In the right panel of Kaggle notebook, navigate to 'Output' -> '/kaggle/working'.")
print("2. Find 'phase3_results.zip' (or 'phase3_results.tar.gz').")
print("3. Click the three dots (...) next to it and select 'Download'.")
